# مائیکروسافٹ ایجنٹ فریم ورک — ایزور اوپن اے آئی (جوابات API)

اس کوڈ سیمپل میں، آپ **مائیکروسافٹ ایجنٹ فریم ورک (MAF)** استعمال کریں گے تاکہ ایک سادہ ایجنٹ بنایا جا سکے جو **ایزور اوپن اے آئی** پر مبنی ہو اور **جوابات API** کا استعمال کرتا ہو۔

> **منتقلی کا نوٹ:** یہ سیمپل پہلے Semantic Kernel کے ساتھ GitHub Models استعمال کرتا تھا۔ اسے مائیکروسافٹ ایجنٹ فریم ورک میں منتقل کیا گیا ہے، اور GitHub Models (جس کی سپورٹ جولائی 2026 میں ختم ہو رہی ہے) کی جگہ ایزور اوپن اے آئی لے چکا ہے، جو جوابات API کی حمایت کرتا ہے۔ MAF میں `OpenAIChatClient`، ایزور اوپن اے آئی کے مستحکم `/openai/v1/` اینڈ پوائنٹ کو ہدف بناتا ہے اور ڈیفالٹ کے طور پر جوابات API استعمال کرتا ہے۔

اس سیمپل کا مقصد وہ اقدامات دکھانا ہے جو بعد میں مختلف ایجنٹ پیٹرنز کو نافذ کرتے وقت اضافی کوڈ سیمپلز میں لاگو کیے جائیں گے۔


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## ضروری پائتھن کے پیکجز درآمد کریں


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## ایک ٹول کی تعریف کرنا

مائیکروسافٹ ایجنٹ فریم ورک میں، ایک **ٹول** ایک سادہ پائتھن فنکشن ہوتا ہے جسے `@tool` کے ساتھ سجایا جاتا ہے جسے ایجنٹ کال کر سکتا ہے۔ نیچے ہم ایک ایسا ٹول ڈیفائن کرتے ہیں جو ایک رینڈم چھٹی کی جگہ واپس کرتا ہے اور پچھلی جگہ کو دہرائے بغیر کام کرتا ہے۔


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## ایجنٹ بنانا

یہاں، ہم `TravelAgent` نامی ایجنٹ بناتے ہیں۔

اس مثال میں، ہم بہت بنیادی ہدایات استعمال کرتے ہیں۔ ان ہدایات کو تبدیل کرنے کے لیے آزاد محسوس کریں تاکہ دیکھ سکیں کہ ایجنٹ کے رویے میں کیسے تبدیلی آتی ہے۔


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## ایجنٹ چلانا

اب ہم ایجنٹ کو چلا سکتے ہیں۔ ہم ایک `AgentSession` بناتے ہیں تاکہ ایجنٹ گفتگو کو مختلف مواقع پر یاد رکھ سکے، پھر دو `user_inputs` بھیجتے ہیں۔ پہلا سفر کے بارے میں پوچھتا ہے؛ دوسرا کہتا ہے کہ صارف کو تجویز پسند نہیں آئی اور دوسرا مانگتا ہے — ایجنٹ سیشن کی تاریخ اور `get_random_destination` ٹول کو استعمال کرتا ہے جواب دینے کے لیے۔

آپ ان پیغامات کو تبدیل کر کے دیکھ سکتے ہیں کہ ایجنٹ مختلف طریقے سے کیسے رد عمل ظاہر کرتا ہے۔ جوابات **اسٹریمنگ** کی صورت میں ایک ایک ٹوکن کے حساب سے دیے جاتے ہیں۔


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
